<h1 align="center">NSDC Data Science Projects</h1>
<h2 align="center">Project: Building a RAG-Powered Research Agent</h2>
<h3 align="center">Name: hellosmallkat </h3>

---


### **Please read before you begin your project**

**Instructions: Google Colab Notebooks:**

Google Colab is a free cloud service. It is a hosted Jupyter notebook service that requires no setup to use, while providing free access to computing resources. We will be using Google Colab for this project.

Certain parts of this project will be completed individually, while other parts are encouraged to be completed with the rest of your team. In order to work within the Google Colab Notebook, please start by clicking on "File" and then "Save a copy in Drive." This will save a copy of the notebook in your personal Google Drive. Each member of your team should work on their personal copy.

Please rename the file to "TITLE - Your Full Name." Once this project is completed, you will be prompted to share your file with the National Student Data Corps (NSDC) Project Leaders.

You can now start working on the project. :)


We'll be using Google Colab for this assignment. This is a Python Notebook environment built by Google that's free for everyone and comes with a nice UI out of the box. For a comprehensive guide, see Colab's official guide [here](https://colab.research.google.com/github/prites18/NoteNote/blob/master/Welcome_To_Colaboratory.ipynb).

Colab QuickStart:
- Notebooks are made up of cells, cells can be either text or code cells. Click the +code or +text button at the top to create a new cell
- Text cells use a format called [Markdown](https://www.markdownguide.org/getting-started/). Cheatsheet is available [here](https://www.markdownguide.org/cheat-sheet/)
- Python code is run/executed in code cells. You can click the play button at the top left of a code block (sometimes hidden in the square brackets) to run the code in that cell. You can also hit shift+enter to run the cell that is currently selected. There is no concurrency since cells run one at a time but you can queue up multiple cells
- Each cell will run code individually but memory is shared across a notebook Runtime. You can think of a Runtime as a code session where everything you create and execute is temporarily stored. This means variables and functions are available between cells if you execute one cell before the other (physical ordering of cells does not matter). This also means that if you delete or change the name of something and re-execute the cell, the old data might still exist in the background. If things aren't making sense, you can always click Runtime -> restart runtime to start over.
- Runtimes will persist for a short period of time so you are safe if you lose connection or refresh the page but Google will shutdown a runtime after enough time has past. Everything that was printed out will remain on the page even if the runtime is disconnected
- Google's Runtimes come preinstalled with all the core python libraries (math, rand, time, etc) as well as common data analysis libraries (numpy, pandas, scikitlearn, matplotlib). Simply run `import numpy as np` in a code cell to make it available

# Introduction

## What Problem Are We Solving?

Imagine you are a researcher trying to understand a new topic. You need to read hundreds of scientific papers from arXiv. But here is the problem:

- Reading hundreds of papers takes weeks
- Large Language Models (LLMs) can hallucinate (make up facts)
- Answers without sources are unverifiable
- You need transparent, grounded responses

## The Solution: RAG (Retrieval-Augmented Generation)

RAG is a technique that:
1. Retrieves relevant papers from arXiv (in this prototype, we'll use paper summaries to keep things fast)
2. Augments your prompts with real content from those papers
3. Generates answers grounded in actual research

## What You Will Build

By the end of this project, you will have:
- A working prototype of a Research Agent that searches arXiv
- A system that creates semantic embeddings
- A vector database that finds relevant passages
- Integration with Claude LLM
- Source highlighting that shows exact text from papers
- An interactive Q&A system

## Architecture Overview

### The Complete Pipeline

```
┌─────────────────────────────────────────────────────────────────┐
│               RAG RESEARCH AGENT ARCHITECTURE                   │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  1. USER INPUT                                                  │
│     "How do transformers work?"                               │
│                    ↓                                            │
│  2. ARXIV SEARCH                                                │
│     • Query arXiv API                                          │
│     • Get relevant paper SUMMARIES       │
│     • Extract metadata                                         │
│                    ↓                                            │
│  3. DOCUMENT PROCESSING (CHUNKING)                              │
│     • Split summaries into smaller chunks                      │
│     • Add metadata to chunks                                   │
│     • Preserve paper references                                │
│                    ↓                                            │
│  4. EMBEDDINGS                                                  │
│     • Convert chunks to vectors                                │
│     • Use Sentence-Transformers                                │
│     • Capture semantic meaning                                 │
│                    ↓                                            │
│  5. VECTOR DATABASE (FAISS)                                     │
│     • Index all embeddings                                     │
│     • Enable fast similarity search                            │
│                    ↓                                            │
│  6. SEMANTIC SEARCH                                             │
│     • Find k most similar chunks                               │
│     • Rank by relevance                                        │
│                    ↓                                            │
│  7. PROMPT BUILDING                                             │
│     • Combine question + retrieved chunks                      │
│     • Add system instructions                                  │
│                    ↓                                            │
│  8. LLM GENERATION (Claude)                                     │
│     • Send prompt to Claude API                                │
│     • Generate grounded response                               │
│                    ↓                                            │
│  9. SOURCE HIGHLIGHTING                   │
│     • Match response to source passages                        │
│     • Highlight exact text from papers                         │
│                    ↓                                            │
│  FINAL OUTPUT                                                   │
│  Answer with [HIGHLIGHTED SOURCE TEXT] + paper links          │
│                                                                 │
└─────────────────────────────────────────────────────────────────┘
```


# Milestone 1: Understanding RAG


### What is RAG?

**RAG = Retrieval-Augmented Generation**

A technique that combines:
1. **Retrieval** - Finding relevant documents
2. **Augmentation** - Adding context to prompts
3. **Generation** - Creating grounded responses

### The Problem RAG Solves

Large Language Models (LLMs) can **hallucinate** - they make up information. A well-designed prompt explicitly tells the LLM to *only* use the provided context, which drastically reduces hallucinations.

```
❌ WITHOUT RAG (Pure LLM):
User: "What did Vaswani et al. say about transformers?"
LLM: "Transformers are... [makes up details]..."
Problems: No sources, may be wrong, unverifiable

✅ WITH RAG (Research Agent):
User: "What did Vaswani et al. say about transformers?"
Agent: "According to the paper, [EXACT QUOTE FROM PAPER]..."
Benefits: Verified, cited, traceable
```


## Reflection Question 1.1

**Question:** In your own words, explain how prompt design affects hallucinations in an LLM.

**Your Answer:**
(Type your answer here)

# Milestone 2: Setup

**Step 1: Install and import libraries**

In [ ]:
# Step 1: Install required packages
import subprocess
import sys

print("📦 Installing required packages...\n")

packages = [
    'sentence-transformers>=2.2.0',
    'faiss-cpu>=1.7.0',
    'anthropic>=0.7.0',
    'numpy>=1.24.0',
    'pandas>=2.0.0'
]

for package in packages:
    package_name = package.split('>')[0].split('=')[0]
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', package, '-q'])
        print(f"✅ {package_name}")
    except Exception as e:
        print(f"⚠️  {package_name} (may already be installed)")

print("\n✅ Installation complete!")

In [ ]:
# Step 2: Import libraries
import os
import json
import numpy as np
import urllib.request
import urllib.parse
from xml.etree import ElementTree as ET
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass
from datetime import datetime
import time
import re

from sentence_transformers import SentenceTransformer
import faiss
from anthropic import Anthropic

print("✅ All imports successful!")

In [ ]:
# Step 3: Configure API — using Colab Secrets (safe & recommended)
# 🔑 Add your Anthropic API key via the Secrets tab in the left sidebar
#    Click the 🔑 key icon → '+ Add new secret' → Name: ANTHROPIC_API_KEY
#    This keeps your key OUT of the notebook and safe from accidental sharing.

from google.colab import userdata

try:
    api_key = userdata.get('ANTHROPIC_API_KEY')
    print("✅ ANTHROPIC_API_KEY loaded from Colab Secrets")
    HAS_API_KEY = True
except Exception:
    print("⚠️  ANTHROPIC_API_KEY not found in Secrets — running in DEMO MODE")
    print("   To fix: open the 🔑 Secrets tab → Add new secret → Name: ANTHROPIC_API_KEY")
    api_key = 'sk-ant-demo'  # Demo mode fallback
    HAS_API_KEY = False


# Milestone 3: Building Components
Now we'll build each component of the Research Agent step-by-step.

In [ ]:
@dataclass
class Paper:
    arxiv_id: str
    title: str
    authors: List[str]
    summary: str
    published: str
    url: str
    pdf_url: str


In [ ]:
# This connects to arXiv API to search for papers
class ArxivConnector:
    """Search and retrieve paper summaries from arXiv"""
    def __init__(self):
        self.base_url = 'http://export.arxiv.org/api/query?'
        self.namespace = {'atom': 'http://www.w3.org/2005/Atom'}

    def search(self, query: str, max_results: int = 5) -> List[Paper]:
        print(f'Searching arXiv for: {query}')
        params = {
            'search_query': query,
            'start': 0,
            'max_results': max_results,
            'sortBy': 'relevance',
            'sortOrder': 'descending'
        }
        url = self.base_url + urllib.parse.urlencode(params)
        try:
            with urllib.request.urlopen(url, timeout=10) as response:
                xml_data = response.read().decode('utf-8')
        except Exception as e:
            print(f'Error: {e}')
            return []

        root = ET.fromstring(xml_data)
        papers = []
        for entry in root.findall('atom:entry', self.namespace):
            try:
                arxiv_id = entry.find('atom:id', self.namespace).text.split('/abs/')[-1]
                title = entry.find('atom:title', self.namespace).text.replace('\n', ' ').strip()
                summary = entry.find('atom:summary', self.namespace).text.replace('\n', ' ').strip()
                published = entry.find('atom:published', self.namespace).text
                authors = []
                for author in entry.findall('atom:author', self.namespace):
                    name = author.find('atom:name', self.namespace).text
                    authors.append(name)
                paper = Paper(
                    arxiv_id=arxiv_id,
                    title=title,
                    authors=authors,
                    summary=summary,
                    published=published,
                    url=f'https://arxiv.org/abs/{arxiv_id}',
                    pdf_url=f'https://arxiv.org/pdf/{arxiv_id}.pdf'
                )
                papers.append(paper)
            except:
                continue
        print(f'Found {len(papers)} papers')
        return papers


### Understanding Chunking
Why do we chunk text? LLMs have context limits, and embedding massive walls of text dilutes the semantic meaning.

* **Chunk Size**: Larger chunks provide more context but might reduce search precision. Smaller chunks are precise but might lose the broader meaning.
* **Overlap**: We overlap chunks (e.g., repeating 100 characters from the previous chunk) to ensure sentences or core concepts aren't accidentally cut in half.

In [ ]:
class DocumentProcessor:
    """Process papers into chunks with metadata"""
    def __init__(self, chunk_size: int = 500, chunk_overlap: int = 100):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap

    def chunk_text(self, text: str) -> List[str]:
        chunks = []
        start = 0
        if len(text) <= self.chunk_size:
            return [text]
        while start < len(text):
            end = min(start + self.chunk_size, len(text))
            chunk = text[start:end]
            chunks.append(chunk)
            start = end - self.chunk_overlap
            if end == len(text):
                break
        return chunks

    def process_papers(self, papers: List[Paper]) -> Tuple[List[str], List[Dict]]:
        all_chunks = []
        all_metadata = []
        for paper in papers:
            text = f'Title: {paper.title}\n\nSummary: {paper.summary}'
            chunks = self.chunk_text(text)
            for chunk_idx, chunk in enumerate(chunks):
                all_chunks.append(chunk)
                all_metadata.append({
                    'paper_id': paper.arxiv_id,
                    'paper_title': paper.title,
                    'paper_url': paper.url,
                    'paper_pdf_url': paper.pdf_url,
                    'paper_authors': paper.authors,
                    'chunk_index': chunk_idx,
                    'chunk_text': chunk
                })
        return all_chunks, all_metadata


### Understanding Embeddings
Embeddings convert text into dense arrays of numbers (vectors) where semantically similar texts are placed closer together in a mathematical space.
Why do they work better than standard keyword search? Because the embedding model has been pre-trained on massive amounts of text to understand *context and meaning*, not just exact word matching. E.g., "Puppy" and "Dog" will be close together even though they share no letters.

In [ ]:
class EmbeddingManager:
    """Manage text embeddings"""
    def __init__(self, model_name: str = 'all-MiniLM-L6-v2'):
        self.model_name = model_name
        self.model = None

    def load_model(self):
        if self.model is None:
            print("📥 Loading embedding model...")
            self.model = SentenceTransformer(self.model_name)

    def encode(self, texts: List[str]) -> np.ndarray:
        self.load_model()
        embeddings = self.model.encode(texts, convert_to_numpy=True)
        return embeddings.astype('float32')

    def encode_query(self, text: str) -> np.ndarray:
        self.load_model()
        embedding = self.model.encode(text, convert_to_numpy=True)
        return embedding.astype('float32').reshape(1, -1)


In [ ]:
class VectorDatabase:
    """FAISS-based vector database for semantic search"""
    def __init__(self, dimension: int = 384):
        self.dimension = dimension
        self.index = faiss.IndexFlatL2(dimension)
        self.is_built = False

    def add(self, vectors: np.ndarray) -> int:
        if vectors.shape[1] != self.dimension:
            raise ValueError(f"Wrong dimension: {vectors.shape[1]} != {self.dimension}")
        self.index.add(vectors)
        self.is_built = True
        return self.index.ntotal

    def search(self, query_vector: np.ndarray, k: int = 3) -> Tuple[np.ndarray, np.ndarray]:
        if not self.is_built:
            return np.array([]), np.array([])
        distances, indices = self.index.search(query_vector, k)
        return distances, indices


In [ ]:
class SourceHighlighter:
    """Match response text to source passages and highlight them"""
    @staticmethod
    def find_relevant_passages(response: str, chunks: List[str]) -> Dict:
        highlighted = {}
        quotes = re.findall(r'["\']([^"\']*)["\'\']', response)
        for chunk in chunks:
            for quote in quotes:
                if quote.lower() in chunk.lower():
                    highlighted[quote] = chunk
                    break
        return highlighted


In [ ]:
class ResearchAgent:
    """Complete RAG Research Agent for scientific papers"""
    def __init__(self, api_key: str):
        self.api_key = api_key
        self.model = "claude-3-5-sonnet-20241022"
        self.arxiv = ArxivConnector()
        self.processor = DocumentProcessor(chunk_size=500, chunk_overlap=100)
        self.embeddings = EmbeddingManager()
        self.vector_db = VectorDatabase()
        self.highlighter = SourceHighlighter()
        self.chunks = []
        self.metadata = []
        self.papers = []
        if api_key.startswith('sk-ant-'):
            self.client = Anthropic(api_key=api_key)
        else:
            self.client = None
            print("⚠️  Running in DEMO MODE")

    def index_papers(self, query: str, max_papers: int = 3) -> bool:
        print("🚀 BUILDING RESEARCH INDEX")
        self.papers = self.arxiv.search(query, max_results=max_papers)
        if not self.papers:
            return False
        self.chunks, self.metadata = self.processor.process_papers(self.papers)
        embeddings = self.embeddings.encode(self.chunks)
        self.vector_db.add(embeddings)
        print("✅ INDEX READY!")
        return True

    def search(self, query: str, k: int = 3) -> List[Dict]:
        if not self.chunks:
            return []
        query_embedding = self.embeddings.encode_query(query)
        distances, indices = self.vector_db.search(query_embedding, k)
        results = []
        for dist, idx in zip(distances[0], indices[0]):
            if idx < len(self.metadata):
                meta = self.metadata[idx].copy()
                meta['similarity'] = 1 / (1 + float(dist))
                results.append(meta)
        return results

    def answer_question(self, question: str, k: int = 3) -> Dict:
        results = self.search(question, k=k)
        if not results:
            return {'answer': 'No relevant papers found.', 'sources': {}}

        context_parts = []
        for idx, result in enumerate(results):
            context_parts.append(
                f"[SOURCE {idx+1}]\nPaper: {result['paper_title']}\nContent: {result['chunk_text']}\n"
            )
        context = "\n---\n".join(context_parts)

        if self.client:
            prompt = f"Answer ONLY based on the provided papers.\nQuestion: {question}\nPapers:\n{context}\nAnswer:"
            message = self.client.messages.create(
                model=self.model, max_tokens=1000,
                messages=[{"role": "user", "content": prompt}]
            )
            answer = message.content[0].text
        else:
            answer = "[DEMO MODE] Generated answer based on sources."

        source_highlights = {}
        for idx, result in enumerate(results):
            source_name = f"SOURCE {idx+1}"
            if source_name in answer:
                sentences = result['chunk_text'].split('.')
                for sentence in sentences:
                    if len(sentence.strip()) > 50:
                        source_highlights[source_name] = {
                            'passage': sentence.strip(),
                            'paper_title': result['paper_title'],
                            'paper_url': result['paper_url'],
                            'authors': result['paper_authors']
                        }
                        break
        return {'answer': answer, 'sources': source_highlights}


# Milestone 4: Running the Agent

Now let's use the agent to search papers and answer questions with source highlighting!

In [ ]:
agent = ResearchAgent(api_key=api_key)
search_query = "transformer attention mechanisms"
agent.index_papers(search_query, max_papers=3)
response = agent.answer_question("How do transformers use attention mechanisms?", k=3)
print(response['answer'])

## Milestone 5: Evaluating our RAG System

How do we know if our agent is doing a good job? Evaluation is a critical step for data scientists. Consider these three dimensions:
1. **Relevance**: Are the retrieved chunks actually answering the user's question, or just matching keywords?
2. **Groundedness (Accuracy)**: Does the final answer rely *only* on the retrieved text, or is the LLM hallucinating?
3. **Consistency**: If you ask the same question multiple times, do you get similar responses?

### Reflection Question
**Question**: What happens if you ask the agent a completely vague query (e.g., "Tell me about computers") or if the retrieved papers are only loosely related?

**Your Answer:**
(Type your answer here)

## Summary and Key Takeaways

### What We Built

We created a **working prototype** of a RAG Research Agent that:
✅ **Searches arXiv** for scientific papers
✅ **Processes paper summaries** into chunks  
✅ **Creates embeddings** for semantic search  
✅ **Builds vector database** for fast retrieval  
✅ **Generates grounded answers** using Claude LLM  

### Moving to Production
To take this prototype to a production environment, you would need to add:
- **Full PDF Parsing**: Extracting and processing the entire text from papers instead of just their summaries.
- **Robust Error Handling**: Managing API rate limits, connection drops, and empty search results gracefully.
- **Database Persistence**: Saving the FAISS vector index to disk so you don't have to rebuild it every session.
- **Evaluation Metrics**: Implementing automated scoring (like RAGAS) to measure relevance and groundedness over time.

**Congratulations!** 🎉 You've successfully built a RAG prototype!